# GPT-2 Inference Engine - Google Colab Setup

This notebook sets up the GPT-2 inference engine on Google Colab with T4 GPU acceleration.

**Runtime:** GPU (T4) | **Estimated time:** 15-20 minutes

## Step 1: Environment Setup

Check GPU availability and install build dependencies.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install build dependencies
!apt-get update && apt-get install -y build-essential cmake git wget curl htop

In [ ]:
# Check CUDA version
!nvcc --version

## Step 2: Clone and Build GGML with CUDA Support

GGML provides the tensor computation backend with CUDA acceleration.

In [ ]:
# Clone GGML
%cd /content
!git clone https://github.com/ggerganov/ggml.git

In [ ]:
# Build GGML with CUDA
%cd /content/ggml
!mkdir -p build && cd build
!cmake .. -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
!make -j$(nproc)

In [ ]:
# Verify GGML build
!ls -la /content/ggml/build/lib/

## Step 3: Clone Inference Engine

Copy your inference engine source files to the Colab environment.

In [ ]:
# Option A: If using Google Drive (mount first)
# from google.colab import drive
# drive.mount('/content/gdrive')

# Option B: If uploading files directly - use the Files panel (left sidebar)
# Or clone from GitHub if hosted:
# !git clone https://github.com/YOUR_USERNAME/inference-engine.git

# For now, we'll create the project structure and upload files
!mkdir -p /content/inference-engine/src

In [ ]:
# Create CMakeLists.txt for the project
cmake_content = """cmake_minimum_required(VERSION 3.16)
project(gpt2_inference)

set(GGML_DIR ${CMAKE_SOURCE_DIR}/../ggml)
include_directories(
    ${GGML_DIR}/include
    ${GGML_DIR}/src
    ${CMAKE_SOURCE_DIR}/src
)

file(GLOB SOURCES src/*.cpp)

add_executable(gpt2 ${SOURCES})

target_link_libraries(gpt2
    ggml
    ggml_static
    cublas
    pthread
)

set_target_properties(gpt2 PROPERTIES
    CUDA_ARCHITECTURES 75
)
"""

with open('/content/inference-engine/CMakeLists.txt', 'w') as f:
    f.write(cmake_content)

print('CMakeLists.txt created')

In [ ]:
# Upload your source files
# Use the Files panel to upload src/model.cpp, src/model.hpp, src/layers.cpp, 
# src/layers.hpp, src/tokenizer.cpp, src/tokenizer.hpp, src/kv_cache.cpp, 
# src/kv_cache.hpp, src/main.cpp

# Or run this cell after uploading via the Files panel
import os

src_dir = '/content/inference-engine/src'
print(f'Source files in {src_dir}:')
for f in os.listdir(src_dir):
    print(f'  {f}')

## Step 4: Acquire GPT-2 Model (GGUF Format)

Download GPT-2 and convert to GGUF format for efficient loading.

In [ ]:
# Install HuggingFace Hub for downloading models
!pip install huggingface_hub safetensors tqdm -q

In [ ]:
# Clone llama.cpp for conversion utilities
%cd /content
!git clone https://github.com/ggerganov/llama.cpp.git

In [ ]:
# Download GPT-2 from HuggingFace
from huggingface_hub import snapshot_download

model_dir = '/content/gpt2-huggingface'
print('Downloading GPT-2 model from HuggingFace...')
snapshot_download(repo_id='gpt2', local_dir=model_dir, local_dir_use_symlinks=False)
print('Done!')

In [ ]:
# Check downloaded files
!ls -la /content/gpt2-huggingface/

In [ ]:
# Convert GPT-2 to GGUF format
%cd /content/llama.cpp
!python3 convert.py /content/gpt2-huggingface/ \
    --outfile /content/gpt2.gguf \
    --outtype f16 \
    --model-type gpt2

In [ ]:
# Verify GGUF file created
!ls -lh /content/gpt2.gguf

## Step 5: Update Source Code for GGUF Loading

The current `load_weights()` needs to be updated to parse GGUF format.

In [ ]:
# Create updated model.cpp with GGUF loading support
# This replaces load_weights() with actual GGUF implementation

model_cpp_content = '''
// ... (rest of your existing model.cpp)

bool GPT2Model::load_weights(const std::string& path) {
    // Try GGUF format first
    struct gguf_context* ctx = gguf_load(path.c_str());
    if (ctx) {
        printf("Loaded GGUF model from: %s\\n", path.c_str());
        
        // Get tensor count
        int n_tensors = gguf_get_n_tensors(ctx);
        printf("Number of tensors: %d\\n", n_tensors);
        
        // Load each tensor
        for (int i = 0; i < n_tensors; i++) {
            const char* name = gguf_get_tensor_name(ctx, i);
            struct ggml_tensor* tensor = ggml_get_tensor(ctx_, name);
            if (tensor) {
                // Read tensor data from file
                // This is handled by ggml_init_from_file
            }
        }
        
        gguf_free(ctx);
        return true;
    }
    
    // Fallback to binary format
    return load_ggml_weights(path);
}
'''

print('GGUF loading code template created')
print('Note: You need to integrate this with your actual model.cpp')

In [ ]:
# Alternative: Use llama.cpp main executable for inference
# This bypasses the need to modify your C++ code

%cd /content/llama.cpp
!mkdir -p build && cd build
!cmake .. -DGGML_CUDA=ON -DLLAMA_BUILD_MAIN=ON -DCMAKE_BUILD_TYPE=Release
!make -j$(nproc)

In [ ]:
# Run inference using llama.cpp (GPT-2 compatible)
%cd /content/llama.cpp/build/bin
!
./llama-cli \
    -m /content/gpt2.gguf \
    -p "Once upon a time" \
    -n 100 \
    --temp 0.8 \
    -t 8 \
    -c 1024 \
    --gpu-layers 32

## Step 6: Build Custom Inference Engine

Build your own inference engine with the updated source code.

In [ ]:
# Build the inference engine
%cd /content/inference-engine
!mkdir -p build && cd build
!cmake .. -DCMAKE_CUDA_ARCHITECTURES=75
!make -j$(nproc)

In [ ]:
# Verify build
!ls -la /content/inference-engine/build/

## Step 7: Run Inference

Execute your inference engine.

In [ ]:
# Run with your inference engine
%cd /content/inference-engine/build
!
./gpt2 "Hello, world!" 50 0.8 40

## Troubleshooting

### Common Issues:

1. **CUDA out of memory**: Reduce batch size or use smaller model
2. **GGUF conversion fails**: Check that GPT-2 files downloaded correctly
3. **Build errors**: Ensure all source files are uploaded to `/content/inference-engine/src/`
4. **Tensor mismatch**: GPT-2 Large (774M) config: 36 layers, 20 heads, 1280 hidden size

### Verify GGUF Model:


In [ ]:
# Check GGUF model metadata
!pip install numpy sentencepiece -q
import struct

def read_gguf_header(path):
    with open(path, 'rb') as f:
        magic = struct.unpack('I', f.read(4))[0]
        print(f'Magic: {hex(magic)}')
        version = struct.unpack('I', f.read(4))[0]
        print(f'Version: {version}')

read_gguf_header('/content/gpt2.gguf')

## File Summary

After uploading your source files, the structure should be:
```
/content/
├── ggml/                    # GGML library (CUDA enabled)
├── llama.cpp/              # Conversion utilities
├── inference-engine/       # Your project
│   ├── src/
│   │   ├── model.cpp
│   │   ├── model.hpp
│   │   ├── layers.cpp
│   │   ├── layers.hpp
│   │   ├── tokenizer.cpp
│   │   ├── tokenizer.hpp
│   │   ├── kv_cache.cpp
│   │   └── kv_cache.hpp
│   ├── build/
│   └── CMakeLists.txt
└── gpt2.gguf              # Model file (~1.5GB)
```

## Next Steps

1. Upload your source files to `/content/inference-engine/src/`
2. Update `model.cpp` to implement GGUF loading
3. Run cells from Step 6 onwards
4. For custom inference logic, modify the model classes

**Note**: The llama.cpp approach (Step 5) provides a working reference implementation while you integrate GGUF loading into your custom engine.